# Comparação pré/pós-cirurgia — pós-operatório inicial

Comparação descritiva dos dados tratados pelo CPAP. Cirurgia em 28/09/2026;
30 dias de recuperação, com `after` a partir de 28/10/2026.

O resumo calculado abaixo destaca 16/10, primeira noite inteira com fita na boca.
As noites parciais de 14 e 15/10 permanecem individualizadas. Não há teste de
significância nem estimativa causal do efeito da cirurgia.

Fontes locais: `data/raw/card/STR.edf` → `data/processed/nights.parquet`;
DATALOG → `sessions`, `signals_2s`, `breaths`, `night_metrics` e
`respiratory_events`. Contexto informado: `data/reference/night_notes.json`.
Antes de atualizar: `uv run sleep-ingest D:`, `uv run sleep-garmin-sync` e `uv run sleep-build`.
O Garmin entra na seção final: FC de repouso, HRV, SpO2 de pulso, respiração e score.
Saídas executadas, tabelas e PNG ficam em `data/reports/` (fora do git).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display, Markdown

from sleep_analysis.db import connect
from sleep_analysis.paths import PROJECT_DIR, PROCESSED_DIR
from sleep_analysis.study import load_study, label_period, load_night_notes

plt.rcParams.update({"figure.figsize": (12, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
REPORT_DIR = PROJECT_DIR / "data/reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
db = connect()
study = load_study()
assert study.surgery_date is not None, "Preencha surgeryDate em config/study.json"
recovery_end = study.surgery_date + pd.Timedelta(days=study.recovery_days_excluded)
nights = db.sql("SELECT * FROM nights ORDER BY night").df()
assert nights.night.is_unique
nights["period"] = label_period(nights, study)
nights = nights.merge(load_night_notes(), on="night", how="left", validate="one_to_one")
post = nights[nights.period == "recovery"].copy()
after = nights[nights.period == "after"].copy()
assert not post.empty, "Sem noites no pós-operatório inicial: importe e reconstrua os dados"
print(f"Resumo diário: {len(nights)} noites com uso, {nights.night.min():%d/%m/%Y} a {nights.night.max():%d/%m/%Y}")
print(f"Pós-operatório inicial: {len(post)} noites; after: {len(after)} noites; início de after: {recovery_end:%d/%m/%Y}")
print(f"Parquet nights atualizado em {pd.Timestamp((PROCESSED_DIR / 'nights.parquet').stat().st_mtime, unit='s', tz='UTC')}")

## Contexto e método

- **Principal:** últimas 60 noites pré a 7,0 cmH₂O, com pelo menos 4 h de uso.
- **Controle na mesma pressão:** todas as noites pré a 6,0 cmH₂O.
- **Sensibilidade:** todas as noites pré a 5,8 cmH₂O, separadamente.

Controles históricos mantêm os usos parciais; não são pareados por época ou hábitos.
Cada noite tem peso igual. O IQR é Q1–Q3, não um intervalo de confiança.
O percentil usa `100 × (n menores + 0,5 × n iguais) / n válidos`.
Percentis altos significam valores maiores, não necessariamente noites melhores.
Não há teste de significância com três noites pós.

### Premissas e denominadores

O dia ResMed vai de meio-dia a meio-dia. AHI/OAI/CAI/HI já são índices por hora
de uso no STR; `mask_events` é contagem, também apresentada por hora.
Os eventos próprios `surge`, `hypopnea30` e `rera_like` são detectados apenas no
sono estimado pelo algoritmo; aqui dividimos suas contagens pelas **horas das
sessões detalhadas**, para comparar por hora de uso. Isso não os torna índices clínicos.
`wake_like` conta janelas de 1 min; `wake_like_bouts` agrupa janelas consecutivas.
Também mostramos horas de sono estimado e cobertura. Ausência de detalhe é NaN,
nunca zero. Os índices originais de `night_metrics` usam horas de sono estimado,
um denominador diferente, e não são misturados aos índices por uso.

In [ ]:
before = nights[nights.period == "before"]
baselines = {
    "Principal 7,0": before[np.isclose(before.pressure, 7.0) & (before.usage_hours >= 4)].tail(study.baseline_nights),
    "Controle 6,0": before[np.isclose(before.pressure, 6.0)],
    "Sensibilidade 5,8": before[np.isclose(before.pressure, 5.8)],
}
assert not baselines["Principal 7,0"].empty
cohorts = pd.DataFrame([
    {"grupo": name, "n": len(frame), "início": frame.night.min(),
     "fim": frame.night.max(), "uso_total_h": frame.usage_hours.sum(),
     "noites_uso_menor_4h": int((frame.usage_hours < 4).sum())}
    for name, frame in baselines.items()
])
display(cohorts.round({"uso_total_h": 2}))
cohorts.to_csv(REPORT_DIR / "05_cohorts.csv", index=False)

## Inventário e cobertura

As notas de fita/uso parcial vêm do relato registrado, não dos sinais do aparelho.
Sem nota, o contexto permanece desconhecido. As noites 28–30/09 foram relatadas
sem uso; dias ausentes do STR não são noites com AHI zero.
O controle de 2025 pode ter resumo diário sem fluxo detalhado no backup.

In [ ]:
coverage = db.sql("""
    WITH s AS (
        SELECT night, count(*) AS sessions, sum(duration_min)/60 AS detail_hours
        FROM sessions GROUP BY night
    ), p AS (
        SELECT night, count(*)*2.0/3600 AS signal_hours,
            count(snore)*2.0/3600 AS snore_hours,
            count(flow_limitation)*2.0/3600 AS flow_lim_hours,
            count(*) FILTER (WHERE snore > 0)*2.0/60 AS snore_minutes,
            count(*) FILTER (WHERE flow_limitation > 0)*2.0/60 AS flow_lim_minutes
        FROM signals_2s GROUP BY night
    ), b AS (
        SELECT night, count(*) AS breath_count FROM breaths GROUP BY night
    )
    SELECT * FROM s FULL OUTER JOIN p USING (night) FULL OUTER JOIN b USING (night)
""").df()
nm = db.sql("SELECT * FROM night_metrics").df()
assert coverage.night.is_unique and nm.night.is_unique
inventory = post.merge(coverage, on="night", how="left", validate="one_to_one")
inventory = inventory.merge(nm[["night", "sleep_hours", "invalid_pct"]], on="night", how="left", validate="one_to_one")
inventory["detail_coverage_pct"] = 100 * inventory.detail_hours / inventory.usage_hours
inventory_columns = ["night", "usage_hours", "pressure", "ahi", "mouth_tape", "partial", "note",
                     "sessions", "detail_hours", "signal_hours", "breath_count", "sleep_hours", "invalid_pct", "detail_coverage_pct"]
display(inventory[inventory_columns].round({key: 2 for key in inventory[inventory_columns].select_dtypes("number").columns}))
inventory[inventory_columns].to_csv(REPORT_DIR / "05_post_inventory.csv", index=False)
for name, frame in baselines.items():
    covered = frame.merge(coverage, on="night", how="left")
    print(f"{name}: {covered.detail_hours.notna().sum()}/{len(frame)} noites com sessões; "
          f"{covered.breath_count.notna().sum()}/{len(frame)} com respirações")

## Métricas por noite

Ronco e limitação de fluxo: **minutos positivos por hora com sinal não nulo**
(0–60 min/h), não contagens de episódios. O limite é `> 0` no sinal do aparelho.
A cobertura do sinal e as horas das sessões são expostas no inventário.
As métricas de fluxo adicionais descrevem o algoritmo existente; `invalid_pct`
e `sleep_hours` ajudam a avaliar sua cobertura. Zero eventos inferidos só é
atribuído quando há respirações, sessões e `sleep_hours > 0`.

In [ ]:
event_counts = db.sql("""
    SELECT night,
        count(*) FILTER (WHERE event_type='surge') AS surge_n,
        count(*) FILTER (WHERE event_type='hypopnea30') AS hypopnea30_n,
        count(*) FILTER (WHERE event_type='rera_like') AS rera_like_n,
        count(*) FILTER (WHERE event_type='wake_like') AS wake_like_n
    FROM respiratory_events GROUP BY night
""").df()
flow = coverage.merge(nm[["night", "sleep_hours", "flow_limited_pct", "stable_pct",
                          "invalid_pct", "wake_like_bouts"]], on="night", how="outer", validate="one_to_one")
flow = flow.merge(event_counts, on="night", how="left", validate="one_to_one")
valid_analysis = (flow.sleep_hours > 0) & (flow.breath_count > 0) & (flow.detail_hours > 0)
for kind in ["surge", "hypopnea30", "rera_like", "wake_like"]:
    flow[f"{kind}_per_use_hour"] = flow[f"{kind}_n"].fillna(0).where(valid_analysis) / flow.detail_hours
flow["wake_bouts_per_use_hour"] = flow.wake_like_bouts.where(valid_analysis) / flow.detail_hours
flow["snore_min_per_hour"] = flow.snore_minutes / flow.snore_hours.where(flow.snore_hours > 0)
flow["flow_lim_min_per_hour"] = flow.flow_lim_minutes / flow.flow_lim_hours.where(flow.flow_lim_hours > 0)
all_data = nights.merge(flow, on="night", how="left", validate="one_to_one")
all_data["mask_events_per_hour"] = all_data.mask_events / all_data.usage_hours.where(all_data.usage_hours > 0)
post_data = all_data[all_data.period == "recovery"]
baseline_data = {name: all_data[all_data.night.isin(frame.night)] for name, frame in baselines.items()}

summary_metrics = {
    "ahi": "AHI (eventos/h de uso)", "oai": "OAI (eventos/h de uso)",
    "cai": "CAI (eventos/h de uso)", "hi": "HI (eventos/h de uso)",
    "leak_50_l_min": "Vazamento p50 (L/min)", "leak_95_l_min": "Vazamento p95 (L/min)",
    "resp_rate_50": "Frequência respiratória p50 (/min)", "tid_vol_50": "Volume corrente p50 (mL)",
    "min_vent_50": "Ventilação minuto p50 (L/min)", "mask_events": "Eventos de máscara (contagem)",
    "mask_events_per_hour": "Eventos de máscara (/h de uso)",
}
flow_metrics = {
    "surge_per_use_hour": "Surges (/h de uso detalhado)",
    "hypopnea30_per_use_hour": "Hypopnea30 (/h de uso detalhado)",
    "rera_like_per_use_hour": "RERA-like (/h de uso detalhado)",
    "wake_like_per_use_hour": "Vigília inferida (min/h de uso detalhado)",
    "wake_bouts_per_use_hour": "Blocos de vigília (/h de uso detalhado)",
    "snore_min_per_hour": "Ronco (min/h observado)",
    "flow_lim_min_per_hour": "Limitação aparelho (min/h observado)",
    "flow_limited_pct": "Respirações limitadas (% no sono estimado)",
    "stable_pct": "Janelas estáveis (% no sono estimado)",
    "sleep_hours": "Sono estimado pelo fluxo (h)", "invalid_pct": "Respirações inválidas (%)",
}

def comparison_tables(metrics):
    summaries, percentiles = [], []
    for name, baseline in baseline_data.items():
        for column, label in metrics.items():
            reference = baseline[column].dropna()
            q = reference.quantile([0.25, 0.5, 0.75])
            row = {"grupo": name, "métrica": label, "n_válido": len(reference),
                   "Q1": q.loc[0.25], "mediana": q.loc[0.5], "Q3": q.loc[0.75]}
            pct = {"grupo": name, "métrica": label, "n_válido": len(reference)}
            for _, night in post_data.iterrows():
                value = night[column]
                date = night.night.strftime("%d/%m/%Y")
                row[date] = value
                pct[date] = (100 * ((reference < value).sum() + 0.5 * (reference == value).sum()) / len(reference)
                             if len(reference) and pd.notna(value) else np.nan)
            summaries.append(row)
            percentiles.append(pct)
    return pd.DataFrame(summaries), pd.DataFrame(percentiles)

summary_comparison, summary_percentiles = comparison_tables(summary_metrics)
flow_comparison, flow_percentiles = comparison_tables(flow_metrics)
for name, frame in [("summary_comparison", summary_comparison), ("summary_percentiles", summary_percentiles),
                    ("flow_comparison", flow_comparison), ("flow_percentiles", flow_percentiles)]:
    frame.to_csv(REPORT_DIR / f"05_{name}.csv", index=False)

## Primeiros achados calculados

Os valores abaixo são descritivos do sono tratado e não atribuem mudanças à cirurgia.

In [ ]:
main = baseline_data["Principal 7,0"]
lines = [f"Linha de base principal: **{len(main)} noites** a 7,0 cmH₂O; "
         f"AHI mediano **{main.ahi.median():.2f}/h** (Q1–Q3 {main.ahi.quantile(.25):.2f}–{main.ahi.quantile(.75):.2f}); "
         f"vazamento p95 mediano **{main.leak_95_l_min.median():.1f} L/min**."]
for _, row in post_data.iterrows():
    lines.append(f"**{row.night:%d/%m}:** {row.usage_hours:.2f} h de uso a {row.pressure:.1f} cmH₂O; "
                 f"AHI {row.ahi:.2f}/h; vazamento p95 {row.leak_95_l_min:.1f} L/min. "
                 f"Contexto: {row.note if pd.notna(row.note) else 'sem nota registrada'}.")
control = baseline_data["Controle 6,0"]
lines.append(f"Controle na mesma pressão (6,0): **{len(control)} noites**; AHI mediano "
             f"**{control.ahi.median():.2f}/h**. Os controles de 6,0 e 5,8 têm resumo diário, "
             "mas nenhum fluxo detalhado disponível.")
focus = post_data[post_data.night == pd.Timestamp("2026-10-16")]
if not focus.empty:
    row = focus.iloc[0]
    lines.append(f"Em 16/10, ronco **{row.snore_min_per_hour:.2f} min/h** contra "
                 f"**{main.snore_min_per_hour.median():.2f} min/h** na referência recente; "
                 f"limitação do aparelho **{row.flow_lim_min_per_hour:.2f}** contra "
                 f"**{main.flow_lim_min_per_hour.median():.2f} min/h**. "
                 "Esses sinais aumentaram; a pressão diferente e o pós-operatório impedem atribuição causal.")
text = "\n\n".join(lines)
display(Markdown(text))
(REPORT_DIR / "05_findings.md").write_text(text + "\n\nComparação exploratória; recuperação incompleta, pressões distintas, fita e uso parcial confundem os resultados.\n", encoding="utf-8")

## Resumo diário: mediana e IQR contra cada noite pós

Q1 e Q3 são os extremos do IQR. `n_válido` é específico da métrica e da linha de base.

In [ ]:
for name in baselines:
    display(Markdown(f"### {name}"))
    display(summary_comparison[summary_comparison.grupo == name].drop(columns="grupo").round(2))

### Percentil de cada noite na distribuição pré

Empates usam o ponto médio. Dados ausentes não recebem percentil.

In [ ]:
display(summary_percentiles.round(1))

## Fluxo: comparação e percentis

NaN nos controles históricos significa detalhe indisponível. As contagens de
vigília são padrões inferidos, não despertares confirmados por EEG.

In [ ]:
for name in baselines:
    display(Markdown(f"### {name}"))
    display(flow_comparison[flow_comparison.grupo == name].drop(columns="grupo").round(2))
display(flow_percentiles.round(1))

## Evolução de AHI e vazamento

Mostramos noites com uso; lacunas são interrupções de observação, sem interpolação.
A faixa sombreada cobre 28/09–27/10. O uso parcial tem marcador triangular;
a primeira noite inteira com fita (16/10) tem estrela.

In [ ]:
POST_STYLES = [("#2463a0", "^"), ("#b5651d", "v"), ("#8c3c70", "*")]

def post_style(i, row):
    color, marker = POST_STYLES[i % len(POST_STYLES)]
    if row.night == pd.Timestamp("2026-10-16"):
        marker = "*"
    return color, marker

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
for ax, column, label in zip(axes, ["ahi", "leak_95_l_min"], ["AHI (eventos/h de uso)", "Vazamento p95 (L/min)"]):
    ax.scatter(nights.night, nights[column], s=12, color="#777777", alpha=0.55, label="Noites com uso")
    ax.axvline(study.surgery_date, color="#333333", linestyle="--", label="Cirurgia 28/09")
    ax.axvspan(study.surgery_date, recovery_end, color="#dfc178", alpha=0.25, label="Pós-operatório inicial")
    for i, (_, row) in enumerate(post_data.iterrows()):
        color, marker = post_style(i, row)
        ax.scatter(row.night, row[column], s=160 if marker == "*" else 65,
                   marker=marker, color=color, zorder=5, label=row.night.strftime("%d/%m"))
    ax.set_ylabel(label)
    ax.grid(axis="y", alpha=0.2)
axes[0].set_title("AHI e vazamento durante o uso do CPAP")
axes[0].legend(ncol=3, fontsize=9, loc="upper left", bbox_to_anchor=(0, 1.02))
axes[-1].set_xlim(nights.night.min() - pd.Timedelta(days=5), max(nights.night.max(), recovery_end) + pd.Timedelta(days=5))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))
axes[-1].set_xlabel(f"Dia ResMed; dados de {nights.night.min():%d/%m/%Y} a {nights.night.max():%d/%m/%Y}")
fig.tight_layout()
fig.savefig(REPORT_DIR / "05_ahi_leak_timeline.png", dpi=160, bbox_inches="tight")
plt.show()

## Distribuição pré e pontos pós

Caixas: Q1–Q3; linha: mediana; bigodes: até 1,5 IQR. Outliers são mantidos.
Cada ponto pós é sobreposto a cada referência; são as mesmas noites, não novas
observações. `n` conta valores válidos por métrica. Não há caixa sem dados.

In [ ]:
def plot_distributions(metrics, filename, title):
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    for ax, (column, label) in zip(axes.flat, metrics.items()):
        ticks = []
        for position, (name, baseline) in enumerate(baseline_data.items(), 1):
            values = baseline[column].dropna()
            ticks.append(f"{name}\nn={len(values)}" + ("; sem detalhe" if values.empty else ""))
            if len(values):
                ax.boxplot([values.to_numpy()], positions=[position], widths=0.45,
                           patch_artist=True, boxprops={"facecolor": "#d9e2eb"},
                           medianprops={"color": "#333333"})
            for i, (_, row) in enumerate(post_data.iterrows()):
                if pd.isna(row[column]):
                    continue
                color, marker = post_style(i, row)
                offset = (i - (len(post_data)-1)/2) * min(0.10, 0.5/max(len(post_data), 1))
                ax.scatter(position + offset, row[column], color=color, marker=marker,
                           s=140 if marker == "*" else 60, zorder=5,
                           label=row.night.strftime("%d/%m") if position == 1 else None)
        ax.set_xticks(range(1, len(ticks)+1), ticks, fontsize=9)
        ax.set_xlim(0.5, len(ticks)+0.5)
        ax.set_ylabel(label)
        ax.grid(axis="y", alpha=0.2)
    axes.flat[0].legend(fontsize=9)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(REPORT_DIR / filename, dpi=160, bbox_inches="tight")
    plt.show()

plot_distributions({key: summary_metrics[key] for key in ["ahi", "leak_95_l_min", "resp_rate_50", "mask_events_per_hour"]},
                   "05_summary_boxplots.png", "Distribuições pré e cada noite do pós-operatório inicial")

In [ ]:
plot_distributions({key: flow_metrics[key] for key in ["surge_per_use_hour", "rera_like_per_use_hour", "snore_min_per_hour", "flow_lim_min_per_hour"]},
                   "05_flow_boxplots.png", "Fluxo durante o uso: distribuições pré e pontos pós")

## Ressalvas

- Apenas três noites na fotografia inicial, duas parciais; não representam uma
  amostra estável do pós-operatório. Usos curtos podem capturar fases diferentes da noite.
- Recuperação incompleta até 27/10. O grupo `after` começa em 28/10.
- Pressão de 6,0 no pós contra 7,0 na referência recente. O controle na mesma
  pressão é antigo, e 5,8 não equivale exatamente a 6,0.
- Fita na boca em 16/10 é variável de confusão: pode contribuir para reduzir
  vazamento. Os dados não isolam o efeito da fita nem o da cirurgia.
- Fluxo/DATALOG incompleto ou corrompido reduz a cobertura; resumo diário e
  sinais detalhados podem ter durações diferentes. Observe os denominadores.
- RERA-like, surges e vigília são estimativas do algoritmo existente, não eventos
  confirmados por exame. Não somar categorias que podem se sobrepor.
- Dados sob CPAP não medem apneia sem tratamento nem fundamentam, sozinhos,
  uma decisão de interromper o aparelho.

## Comparação definitiva — a partir de 28/10/2026

**Placeholder:** após importar noites suficientes do grupo `after`, repetir as
mesmas referências e revisar pressão, duração e fita antes da comparação.
O código abaixo apenas informa a disponibilidade; não promove noites de
recuperação para `after`.

## Garmin no pós-operatório

FC de repouso, HRV noturna, SpO2 de pulso, respiração média e score, na mesma comparação das seções anteriores. A SpO2 é a do relógio, não uma oximetria clínica. HRV só existe a partir de maio/2026, então `n_válido` pode ser menor que o das métricas do CPAP. Sem a tabela `garmin_nights`, a seção avisa e segue.


In [ ]:
tables = set(db.sql("SELECT table_name FROM information_schema.tables").df().table_name)
if "garmin_nights" not in tables:
    print("Sem garmin_nights. Rode sleep-garmin-sync e sleep-build para incluir o relógio.")
else:
    garmin_nights = db.sql("""
        SELECT night, resting_heart_rate, avg_overnight_hrv, spo2_avg, spo2_lowest,
               respiration_avg, sleep_score, sleep_hours
        FROM garmin_nights
    """).df()
    garmin_nights["night"] = pd.to_datetime(garmin_nights.night)
    garmin_columns = {
        "resting_heart_rate": "garmin_resting_hr",
        "avg_overnight_hrv": "garmin_hrv",
        "spo2_avg": "garmin_spo2_avg",
        "spo2_lowest": "garmin_spo2_lowest",
        "respiration_avg": "garmin_respiration",
        "sleep_score": "garmin_sleep_score",
        "sleep_hours": "garmin_sleep_hours",
    }
    garmin_metrics = {
        "garmin_resting_hr": "FC de repouso Garmin (bpm)",
        "garmin_hrv": "HRV noturna Garmin (ms)",
        "garmin_spo2_avg": "SpO2 média Garmin (%)",
        "garmin_spo2_lowest": "SpO2 mínima Garmin (%)",
        "garmin_respiration": "Respiração média Garmin (/min)",
        "garmin_sleep_score": "Score de sono Garmin",
        "garmin_sleep_hours": "Sono Garmin (h)",
    }
    garmin_joined = all_data.merge(
        garmin_nights.rename(columns=garmin_columns)[["night", *garmin_columns.values()]],
        on="night", how="left", validate="one_to_one",
    )
    saved_baseline, saved_post = baseline_data, post_data
    baseline_data = {name: garmin_joined[garmin_joined.night.isin(frame.night)] for name, frame in baselines.items()}
    post_data = garmin_joined[garmin_joined.period == "recovery"]
    garmin_comparison, garmin_percentiles = comparison_tables(garmin_metrics)
    plot_distributions(
        {key: garmin_metrics[key] for key in ["garmin_resting_hr", "garmin_hrv", "garmin_spo2_avg", "garmin_respiration"]},
        "05_garmin_boxplots.png",
        "Garmin: distribuições pré e pontos do pós-operatório inicial",
    )
    baseline_data, post_data = saved_baseline, saved_post
    for name, frame in (("garmin_comparison", garmin_comparison), ("garmin_percentiles", garmin_percentiles)):
        frame.to_csv(REPORT_DIR / f"05_{name}.csv", index=False)
    for name in baselines:
        display(Markdown(f"### {name}"))
        display(garmin_comparison[garmin_comparison.grupo == name].drop(columns="grupo").round(2))
    display(garmin_percentiles.round(1))
    focus = garmin_joined[garmin_joined.night == pd.Timestamp("2026-10-16")]
    reference = baseline_data["Principal 7,0"]
    if not focus.empty and reference.night.isin(garmin_nights.night).any():
        row = focus.iloc[0]
        matched = garmin_joined[garmin_joined.night.isin(reference.night)]
        resting = f"{row.garmin_resting_hr:.0f}" if pd.notna(row.garmin_resting_hr) else "sem leitura"
        hrv = f"{row.garmin_hrv:.0f}" if pd.notna(row.garmin_hrv) else "sem leitura"
        display(Markdown(
            f"Em 16/10, FC de repouso **{resting} bpm** "
            f"(mediana pré recente **{matched.garmin_resting_hr.median():.0f}**) e HRV "
            f"**{hrv} ms** (mediana **{matched.garmin_hrv.median():.0f}**). "
            "Descritivo do pós-operatório; não atribui a mudança à cirurgia."
        ))


In [ ]:
print(f"Grupo after disponível: {len(after)} noites a partir de {recovery_end:%d/%m/%Y}.")
if after.empty:
    print("Aguardando novas noites após a recuperação para a comparação definitiva.")
else:
    display(after[["night", "usage_hours", "pressure", "mouth_tape", "partial"]])
db.close()